# Preprocesamiento del Retail Sales Dataset

**Grupo 2** — Angel Espin · Carlos Ramirez

Este cuaderno documenta la limpieza y derivacion de atributos
realizada sobre el dataset original de Kaggle.

---

## Proceso:
1. Descarga del CSV original (espejo publico en GitHub)
2. Validacion de calidad (nulos, duplicados, consistencia aritmetica)
3. Derivacion de nuevos atributos para el dashboard
4. Exportacion del dataset limpio

In [1]:
# 1. Imports y descarga del dataset original
import pandas as pd
from pathlib import Path

BASE = Path.cwd()
RAW_URL = "https://raw.githubusercontent.com/RISHIshrivas/Retail-Sales-data-analysis/main/retail_sales_dataset.csv"
CLEAN = BASE / "retail_sales_clean.csv"

df = pd.read_csv(RAW_URL)
df["Date"] = pd.to_datetime(df["Date"])
print("Dimensiones:", df.shape)
df.head()

Dimensiones: (1000, 9)


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [2]:
# 2. Validacion de calidad de datos
print("Nulos por columna:\n", df.isnull().sum(), "\n")
print("Filas duplicadas:", df.duplicated().sum())
print("Transaction ID unicos:", df["Transaction ID"].nunique())
print("Customer ID unicos:", df["Customer ID"].nunique())

# Consistencia aritmetica: Total Amount == Quantity * Price per Unit
inconsistentes = (df["Quantity"] * df["Price per Unit"] != df["Total Amount"]).sum()
print("Filas con Total != Quantity*Price:", inconsistentes)

# Rango temporal
print("Fecha min:", df["Date"].min().date(), "| Fecha max:", df["Date"].max().date())
print("Transacciones por anio:\n", df["Date"].dt.year.value_counts())

Nulos por columna:
 Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64 

Filas duplicadas: 0
Transaction ID unicos: 1000
Customer ID unicos: 1000
Filas con Total != Quantity*Price: 0
Fecha min: 2023-01-01 | Fecha max: 2024-01-01
Transacciones por anio:
 Date
2023    998
2024      2
Name: count, dtype: int64


In [3]:
# 3. Derivacion de atributos nuevos
MESES = {1:"01-Enero",2:"02-Febrero",3:"03-Marzo",4:"04-Abril",
         5:"05-Mayo",6:"06-Junio",7:"07-Julio",8:"08-Agosto",
         9:"09-Septiembre",10:"10-Octubre",11:"11-Noviembre",12:"12-Diciembre"}
DIAS  = {0:"1-Lunes",1:"2-Martes",2:"3-Miercoles",3:"4-Jueves",
         4:"5-Viernes",5:"6-Sabado",6:"7-Domingo"}

df["Anio"]      = df["Date"].dt.year
df["Mes"]       = df["Date"].dt.month
df["NombreMes"] = df["Mes"].map(MESES)
df["Trimestre"] = "T" + df["Date"].dt.quarter.astype(str)
df["DiaSemana"] = df["Date"].dt.dayofweek.map(DIAS)
df["TipoDia"]   = df["Date"].dt.dayofweek.apply(
    lambda d: "Fin de semana" if d >= 5 else "Entre semana"
)
df["GrupoEdad"] = pd.cut(df["Age"], bins=[18,25,35,45,55,65],
                         labels=["18-25","26-35","36-45","46-55","56-65"],
                         include_lowest=True, right=True).astype(str)
df["NivelPrecio"] = df["Price per Unit"].apply(
    lambda p: "Bajo (<=50)" if p <= 50 else "Alto (>=300)"
)

df.to_csv(CLEAN, index=False, encoding="utf-8-sig")
print("CSV limpio guardado en:", CLEAN)
print("Atributos finales (", len(df.columns), "):")
print(list(df.columns))
df.head()

CSV limpio guardado en: C:\Users\Espin\Desktop\maestria\proyecto\dashboard_streamlit\retail_sales_clean.csv
Atributos finales ( 17 ):
['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount', 'Anio', 'Mes', 'NombreMes', 'Trimestre', 'DiaSemana', 'TipoDia', 'GrupoEdad', 'NivelPrecio']


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount,Anio,Mes,NombreMes,Trimestre,DiaSemana,TipoDia,GrupoEdad,NivelPrecio
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150,2023,11,11-Noviembre,T4,5-Viernes,Entre semana,26-35,Bajo (<=50)
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000,2023,2,02-Febrero,T1,1-Lunes,Entre semana,26-35,Alto (>=300)
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30,2023,1,01-Enero,T1,5-Viernes,Entre semana,46-55,Bajo (<=50)
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500,2023,5,05-Mayo,T2,7-Domingo,Fin de semana,36-45,Alto (>=300)
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100,2023,5,05-Mayo,T2,6-Sabado,Fin de semana,26-35,Bajo (<=50)


In [4]:
# 4. Resumen final del dataset limpio
print("=" * 50)
print("RESUMEN DEL DATASET LIMPIO")
print("=" * 50)
print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
print(f"Ingresos totales: ${df['Total Amount'].sum():,.0f}")
print(f"Ticket promedio: ${df['Total Amount'].mean():,.2f}")
print(f"Unidades vendidas: {df['Quantity'].sum():,}")
print(f"Categorias: {df['Product Category'].nunique()}")
print(f"Rango edad: {df['Age'].min()} - {df['Age'].max()}")
print(f"Rango fechas: {df['Date'].min().date()} a {df['Date'].max().date()}")
print(f"Valores nulos: {df.isnull().sum().sum()}")
print(f"Filas duplicadas: {df.duplicated().sum()}")

RESUMEN DEL DATASET LIMPIO
Filas: 1,000
Columnas: 17
Ingresos totales: $456,000
Ticket promedio: $456.00
Unidades vendidas: 2,514
Categorias: 3
Rango edad: 18 - 64
Rango fechas: 2023-01-01 a 2024-01-01
Valores nulos: 0
Filas duplicadas: 0
